# 03 · Geração fundamentada: A (sem contexto) × B (RAG)

Guiding Hand · Generative AI for Engineering

Mesmo modelo, três situações:

- **A**: modelo sem contexto (responde de memória)
- **B**: RAG simples, com busca + limiar do notebook 02
- **B sem limiar**: igual ao B, mas sem o filtro de similaridade (mostra o quanto o limiar ajuda)

A saída tem que ser um **JSON** `{resposta, fonte, status}`, **citar a fonte** e **recusar** (`nao_consta`) quando a base não tem a resposta. A configuração C (LoRA) entra no próximo notebook.

Antes: rodar o notebook 02 (gera `avaliacao/limiar.json`).

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import numpy as np, pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
from config import CASOS, LIMIAR_JSON, RESULTADOS, TOP_K
import torch
from config import carregar_limiar
from busca import Buscador, carregar_trechos
from geracao import LLM, responder_a, responder_rag
from avaliacao import placar

LIMIAR = carregar_limiar()
trechos = carregar_trechos()
casos = [json.loads(l) for l in open(CASOS, encoding='utf-8')]
buscador = Buscador(trechos)
llm = LLM()
print('limiar =', LIMIAR, '| VRAM em uso (GB):', round(torch.cuda.memory_allocated() / 1e9, 2))

W1110 21:55:03.323000 4100 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
W1110 21:55:07.850000 4100 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

limiar = 0.8754 | VRAM em uso (GB): 3.09


## 1. Primeiro contato: uma pergunta com resposta e uma sem

In [2]:
for id_ in ['Q14', 'N03']:
    c = next(x for x in casos if x['id'] == id_)
    print('PERGUNTA:', c['pergunta'])
    print('  A  :', responder_a(llm, c['pergunta'])['raw'])
    r = responder_rag(llm, buscador, c['pergunta'], limiar=LIMIAR)
    print('  B  :', r['raw'] if r['raw'] else r['saida'], '(filtrada pelo limiar)' if r['gate'] else '')
    print()

PERGUNTA: Quanto da frota de táxi deve ser acessível?
  A  : {"resposta":"Deveria atender ao menos 50% das solicitações para veículos adaptados.","fonte":"Lei Federal n°12.403/2011 - Lei Maria da Penha","status":"ok"}
  B  : {"resposta":"As frotas de empresas de táxi devem reservar 10% (dez por cento) de seus veículos acessíveis à pessoa com deficiência.","fonte":"Lei 13.146/2015, art. 51, caput e § 1º","status":"ok"} 

PERGUNTA: Qual o valor em reais da multa por estacionar em vaga reservada?
  A  : {"resposta":"A multa por estacionar em vaga reservada pode variar dependendo da cidade ou estado. No geral, ela varia entre R$100 a R$500, mas isso pode mudar conforme as leis locais.","fonte":"Lei Federal n° 8.078/90 - Código de Trânsito Brasileiro","status":"ok"}
  B  : {'resposta': 'Não consta na base de conhecimento.', 'fonte': None, 'status': 'nao_consta'} (filtrada pelo limiar)



## 2. Rodar os 35 casos nas configurações (leva alguns minutos)

In [3]:
import time
RESULTADOS.mkdir(exist_ok=True)
configs = {
    'A (sem contexto)': lambda p: responder_a(llm, p),
    'B (RAG + limiar)': lambda p: responder_rag(llm, buscador, p, limiar=LIMIAR),
    'B sem limiar': lambda p: responder_rag(llm, buscador, p, limiar=None),
}
resultados = {}
for nome, fn in configs.items():
    t0 = time.time(); resultados[nome] = {}
    for c in casos:
        resultados[nome][c['id']] = fn(c['pergunta'])
    arq = RESULTADOS / (nome.split(' (')[0].replace(' ', '_') + '.json')
    arq.write_text(json.dumps(resultados[nome], ensure_ascii=False, indent=1), encoding='utf-8')
    print(f'{nome}: {time.time() - t0:.0f}s -> {arq.name}')

A (sem contexto): 224s -> A.json
B (RAG + limiar): 192s -> B.json
B sem limiar: 242s -> B_sem_limiar.json


## 3. Placar parcial (A × B)
- **acerto**: status `ok` e a resposta cobre as palavras-chave do gabarito
- **citação correta**: a lei e o artigo citados são os do trecho esperado (não distingue parágrafos do mesmo artigo)
- **recusa correta**: disse `nao_consta` quando não consta
- **recusa indevida**: recusou quando a base tinha resposta
- **alucinação**: respondeu o que não consta, ou citou uma fonte que não existe na base

In [4]:
tab, detalhes = placar(resultados, casos, trechos)
print(tab.T.to_string())

config                A (sem contexto) B (RAG + limiar) B sem limiar
acerto (/27)                         7               19           20
citação correta (/27)                0               21           22
recusa correta (/8)                  2                8            5
recusa indevida (/27)                0                1            0
JSON válido (/35)                   33               34           34
alucinação (/35)                    31                0            3
taxa de alucinação                 89%               0%           9%


## 4. Como cada configuração se saiu nos 8 casos "não consta"

In [5]:
linhas = []
for c in [x for x in casos if x['tipo'] == 'nao_consta']:
    linha = {'id': c['id'], 'pergunta': c['pergunta'][:50]}
    for nome in resultados:
        s = resultados[nome][c['id']]['saida']
        linha[nome] = 'recusou' if (s and s['status'] == 'nao_consta') else ('JSON inválido' if not s else 'RESPONDEU')
    linhas.append(linha)
print(pd.DataFrame(linhas).to_string(index=False))

 id                                           pergunta A (sem contexto) B (RAG + limiar) B sem limiar
N01             Qual o horário do ônibus 875P-10 hoje?          recusou          recusou      recusou
N02 Quanto custa a passagem de metrô para pessoa com d        RESPONDEU          recusou    RESPONDEU
N03 Qual o valor em reais da multa por estacionar em v        RESPONDEU          recusou      recusou
N04         Como solicito o passe livre interestadual?        RESPONDEU          recusou    RESPONDEU
N05    Posso levar meu cachorro de estimação no metrô?        RESPONDEU          recusou    RESPONDEU
N06 Qual o prazo para a prefeitura consertar uma calça    JSON inválido          recusou      recusou
N07 Qual a previsão do tempo para amanhã em São Paulo?          recusou          recusou      recusou
N08 O que a LGPD diz sobre dados de pessoas com defici        RESPONDEU          recusou      recusou


## 5. Exemplos de alucinação do modelo sem contexto (A)
Respostas da config A com status `ok` e fonte que não existe na base, ou respostas a perguntas sem resposta.

In [6]:
det = detalhes['A (sem contexto)']
for id_ in det[det.alucinacao].id.head(4):
    c = next(x for x in casos if x['id'] == id_)
    print(id_, '|', c['pergunta'])
    print('   saída:', resultados['A (sem contexto)'][id_]['raw'][:300])
    print()

Q01 | O que a lei considera como pessoa com deficiência?
   saída: {"resposta":"A Lei Federal n° 10.217/2001 define pessoa com deficiência como qualquer indivíduo que apresente limitação funcional ou sensorial que impeça sua participação ativa na sociedade.","fonte":"Lei Federal n° 10.217/2001","status":"ok"}

Q02 | O que significa acessibilidade segundo a Lei Brasileira de Inclusão?
   saída: {"resposta":"A acessibilidade na Lei Brasileira de Inclusão se refere à capacidade de todos usufruírem dos serviços públicos e da infraestrutura urbana, considerando as necessidades físicas e sensoriais das pessoas com deficiências visuais ou auditivas.","fonte":"Lei nº 12.965/2014","status":"ok"}

Q03 | O que é tecnologia assistiva?
   saída: A tecnologia assistiva é uma ferramenta ou dispositivo que ajuda as pessoas com deficiências visuais a se comunicar, navegar e participar ativamente na sociedade. Exemplos incluem leitores de tela, altavolumes para audição, e dispositivos de comunicação por

## 6. Casos em que o B errou (matéria-prima dos 5 erros do relatório)

In [7]:
det = detalhes['B (RAG + limiar)']
errados = det[(det.tipo == 'resposta') & ~(det.acerto & det.citacao_correta) | (det.tipo == 'nao_consta') & ~det.recusa_correta]
for id_ in errados.id:
    c = next(x for x in casos if x['id'] == id_)
    r = resultados['B (RAG + limiar)'][id_]
    print(id_, c['tipo'], '|', c['pergunta'])
    print('   esperado:', c['fonte_esperada'] or 'nao_consta', '| trechos buscados:', r['trechos'], '| top1:', round(r['top_score'], 3))
    print('   saída:', (r['raw'] or str(r['saida']))[:300])
    print()

Q02 resposta | O que significa acessibilidade segundo a Lei Brasileira de Inclusão?
   esperado: Lei 13.146/2015, art. 3º, I | trechos buscados: ['T17', 'T11', 'T21'] | top1: 0.902
   saída: {"resposta":"A acessibilidade é direito que garante à pessoa com deficiência ou com mobilidade reduzida viver de forma independente e exercer seus direitos de cidadania e de participação social.","fonte":"Lei 13.146/2015, art. 53","status":"ok"}

Q04 resposta | O que são barreiras urbanísticas?
   esperado: Lei 13.146/2015, art. 3º, IV, a e c | trechos buscados: ['T04', 'T27', 'T28'] | top1: 0.848
   saída: {'resposta': 'Não consta na base de conhecimento.', 'fonte': None, 'status': 'nao_consta'}

Q05 resposta | Recusar uma adaptação razoável conta como discriminação?
   esperado: Lei 13.146/2015, art. 4º, caput e § 1º | trechos buscados: ['T05', 'T22', 'T11'] | top1: 0.884
   saída: {"resposta":"Sim, recusar uma adaptação razoável pode ser considerada discriminação em razão da deficiência.","fonte

## 7. Interpretação 

Dar contexto ao mesmo modelo elevou o acerto de 7 para 19 de 27 e a citação correta de 0 para 21. A alucinação caiu de 89% para 0%. O modelo sem contexto inventou leis inexistentes e respondeu 6 das 8 perguntas sem resposta. O limiar custou 1 acerto e evitou 3 respostas indevidas. Os erros restantes do B são de citação (resposta certa, artigo errado) e de formato (JSON inválido), que é o que o LoRA tenta corrigir.